# Regime-Based Forecasting Experiments Plan

This notebook defines a practical and research-oriented experimental design to compare three forecasting models across multiple demand regimes:

- `SARIMAX`
- `HURDLE`
- `TSB` (Teunter-Syntetos-Babai)

Datasets to use:
- `M5 / Walmart`
- `Favorita`
- `Amazon 2023`

Target setting:
- 10 to 15 products per dataset
- products selected to cover different demand regimes
- experiments organized under a regime-aware forecasting perspective


## 1. Experimental Overview

### Research Goal

Determine which forecasting model performs better depending on the observed demand regime, including:

- high demand / stable demand
- medium demand / variable demand
- low demand
- intermittent demand with many zeros

### Core Hypothesis

A single forecasting model is unlikely to perform best for all products. Instead:

- `SARIMAX` should work better for stable, dense, and high-demand series.
- `HURDLE` should work better for zero-heavy series with explanatory patterns in occurrence and size.
- `TSB` should work better for truly intermittent, low-rotation, sparse series.

### High-Level Experimental Logic

1. Characterize demand behavior.
2. Detect and label demand regimes.
3. Compare all models without regime adaptation.
4. Compare all models inside each regime.
5. Build and evaluate a hybrid regime-based strategy.


## 2. Experiment-by-Experiment Design

### Experiment 1: Demand Characterization

**Objective**

Describe product-level demand behavior using interpretable statistics and identify which products are suitable for dense-demand models versus intermittent-demand models.

**Inputs**

- Daily sales time series per product.
- 10 to 15 selected products from each dataset.

**Features to compute**

- mean demand
- standard deviation
- percentage of zero demand (`zero_rate`)
- `ADI` (Average Demand Interval)
- `CV²` (squared coefficient of variation over non-zero demand)

**Expected contribution**

This experiment creates the descriptive basis for all later experiments and justifies why regime-aware modeling is necessary.

**Implementation idea in Python**

For each series:
- compute descriptive demand indicators
- store them in one summary table
- use that table to pre-select products for downstream experiments

**Pseudocode**
```python
for product_id, df in series_map.items():
    y = df['sales'].values
    mean_demand = y.mean()
    std_demand = y.std()
    zero_rate = (y == 0).mean()
    adi = len(y) / max((y > 0).sum(), 1)
    nz = y[y > 0]
    cv2 = ((nz.std() / max(nz.mean(), 1e-6)) ** 2) if len(nz) > 1 else np.nan
```

---

### Experiment 2: Regime Detection

**Objective**

Assign each product or period to one demand regime, such as:

- low demand regime
- medium demand regime
- high demand regime
- intermittent demand regime

**Design options**

Two complementary approaches are recommended:

1. **Product-level regime assignment** using summary indicators.
2. **Time-varying regime detection** using rolling windows or Markov-style latent states.

**Recommended practical regime rules**

- `stable/high`: low zero rate, low ADI, lower CV², higher mean
- `medium/variable`: moderate mean, moderate-to-high variance, not necessarily many zeros
- `low demand`: low mean demand, sparse signal, may still be non-intermittent
- `intermittent`: high zero rate and high ADI

**Implementation idea in Python**

- start with rule-based thresholds for reproducibility
- optionally compare with Markov or clustering-based regime detection

**Pseudocode**
```python
def assign_regime(mean_demand, zero_rate, adi, cv2):
    if zero_rate >= 0.50 or adi >= 1.32:
        return 'intermittent'
    if mean_demand >= high_threshold and zero_rate < 0.10:
        return 'high_stable'
    if mean_demand <= low_threshold:
        return 'low_demand'
    return 'medium_variable'
```

---

### Experiment 3: Baseline Benchmark

**Objective**

Compare `SARIMAX`, `HURDLE`, and `TSB` on the selected products without using regime-based adaptation.

**Why this matters**

This provides a global benchmark and answers the question: if one model had to be used uniformly, which one would be strongest overall?

**Implementation idea in Python**

- use the same train/test split for all products
- run all three models on every selected product
- compute both standard accuracy metrics and intermittent-demand metrics

**Recommended split**

- rolling-origin evaluation when possible
- or final 20% / 365-day holdout as a practical starting point

**Expected interpretation**

- `SARIMAX` may dominate global metrics if many products are dense.
- `HURDLE` and `TSB` may look weaker globally but stronger in sparse subsets.

---

### Experiment 4: Regime-Based Comparison

**Objective**

Evaluate model performance separately inside each regime.

**Why this matters**

This is likely the most important experiment for the paper because it directly supports the regime-aware argument.

**Implementation idea in Python**

- assign each selected series to a regime
- group results by regime and model
- compare average and median performance inside regime groups

**Expected interpretation**

- `SARIMAX` should perform best in stable/high-demand groups
- `HURDLE` should perform well in zero-heavy medium/variable groups
- `TSB` should outperform on intermittent and very-low-demand series

---

### Experiment 5: Hybrid Strategy

**Objective**

Design a rule-based or regime-based model selection strategy and compare it against single-model baselines.

**Hybrid strategy idea**

- use `SARIMAX` for stable/high demand
- use `HURDLE` for many zeros or mixed sparse behavior
- use `TSB` for clear intermittent demand

**Implementation idea in Python**

Build a meta-selector that uses regime features to choose the forecasting model before fitting.

**Rule-based example**
```python
def select_model(stats):
    if stats['zero_rate'] >= 0.50 or stats['adi'] >= 1.32:
        return 'TSB'
    if stats['zero_rate'] >= 0.20:
        return 'HURDLE'
    return 'SARIMAX'
```

**Expected contribution**

This experiment translates the scientific findings into an actionable forecasting policy for operations or deployment.


## 3. Metrics

For intermittent-demand forecasting, standard metrics alone are not enough.

### Core error metrics

- `MAE`: easy to interpret and robust
- `RMSE`: penalizes large errors
- `WAPE`: useful for aggregated interpretability
- `MASE`: important because it is scale-free and good for comparing across products

### Intermittent-demand-specific diagnostics

- `bias`: useful to detect systematic overforecasting or underforecasting
- `zero-prediction calibration`: how well the model handles no-demand periods
- `direction accuracy`: useful when demand moves from zero to non-zero
- `service-level style metrics`: optional if inventory interpretation matters

### Recommended paper metric set

For the paper, I recommend reporting:

- `MAE`
- `RMSE`
- `WAPE`
- `MASE`
- `bias`
- `zero_rate_train` and `ADI` as regime descriptors

### Why these are appropriate

- `MAE` and `WAPE` are easy to explain.
- `MASE` enables fair comparison across products and datasets.
- `bias` is critical for intermittent-demand settings, because some models systematically overpredict sparse demand.


## 4. Visualizations

### Experiment 1: Demand Characterization

Generate:
- histogram of mean demand
- histogram of zero rate
- scatter plot: `ADI` vs `CV²`
- scatter plot: mean demand vs zero rate
- small multiples of raw product time series

### Experiment 2: Regime Detection

Generate:
- regime assignment heatmap by product
- timeline plot with regime labels
- rolling zero-rate plot
- clustered scatter plot of regime groups

### Experiment 3: Baseline Benchmark

Generate:
- boxplots of `MAE` / `WAPE` by model
- bar chart of average error by model
- actual vs predicted plots for representative products

### Experiment 4: Regime-Based Comparison

Generate:
- grouped bar chart: model performance inside each regime
- violin or boxplot of errors by regime and model
- regime-specific forecast examples

### Experiment 5: Hybrid Strategy

Generate:
- flowchart of rule-based selector
- hybrid vs best single model comparison
- confusion-style table showing how often each model is selected by regime

### Exact plots recommended for the paper

1. `ADI` vs `CV²` regime map
2. Example stable/high-demand product with `SARIMAX`
3. Example medium/variable product with `HURDLE`
4. Example intermittent product with `TSB`
5. Regime-by-model performance comparison bar chart
6. Final hybrid-strategy results table or figure


## 5. Code Architecture

A clean and modular project structure could look like this:

```text
src/
  data_loaders/
    load_m5.py
    load_favorita.py
    load_amazon.py

  features/
    demand_statistics.py
    regime_features.py

  models/
    sarimax_model.py
    hurdle_model.py
    tsb_model.py
    regime_forecast_engine.py

  evaluation/
    metrics.py
    rolling_validation.py

  experiments/
    run_experiment_01_characterization.py
    run_experiment_02_regime_detection.py
    run_experiment_03_baseline_benchmark.py
    run_experiment_04_regime_comparison.py
    run_experiment_05_hybrid_strategy.py

notebooks/
  paper_experiments/
    23_regime_forecasting_experiments_plan.ipynb
```

### Suggested functions/modules

**Demand statistics**
```python
def compute_demand_statistics(y):
    ...
```

**Regime assignment**
```python
def assign_regime_from_stats(stats):
    ...
```

**Model runners**
```python
def fit_and_forecast_sarimax(y_train, X_train, X_test):
    ...

def fit_and_forecast_hurdle(y_train, X_train, X_test):
    ...

def fit_and_forecast_tsb(y_train, horizon):
    ...
```

**Evaluation**
```python
def evaluate_forecast(y_true, y_pred):
    ...
```

**Hybrid selector**
```python
def select_model_by_regime(stats):
    ...
```

### Recommended implementation roadmap

1. Standardize loaders across the 3 datasets.
2. Build a common daily-series format: `date`, `sales`, `price` if available.
3. Compute descriptive demand statistics.
4. Assign regimes using reproducible thresholds.
5. Run all three models per product.
6. Store all results in one long-format table.
7. Aggregate performance by regime.
8. Implement hybrid selector.
9. Compare hybrid vs single-model baselines.


## 6. Expected Findings

### Regime-specific expectations

- **High demand / stable demand**
  - expected best model: `SARIMAX`
  - reason: temporal continuity and seasonality are easier to model with dense observations

- **Medium demand / variable demand**
  - expected best model: `HURDLE` or sometimes `SARIMAX`
  - reason: demand occurrence and size can begin to decouple

- **Low demand but not strongly intermittent**
  - expected best model: `HURDLE`
  - reason: zeros become more important, but there may still be useful covariate structure

- **Intermittent demand with many zeros**
  - expected best model: `TSB`
  - reason: TSB directly models occurrence probability and non-zero demand size, which is exactly the sparse-demand use case

### Paper-level conclusion expected

The expected final conclusion is not that one model wins globally, but that **model performance depends on the demand regime** and that a **hybrid regime-based forecasting strategy** is more defensible than a uniform forecasting policy.


## Final Table Structure for the Paper

### Table A: Product and regime characterization

| dataset | product_id | mean_demand | std_demand | zero_rate | ADI | CV² | regime |
|---------|------------|-------------|------------|-----------|-----|-----|--------|

### Table B: Global baseline benchmark

| dataset | product_id | model | MAE | RMSE | WAPE | MASE | bias |
|---------|------------|-------|-----|------|------|------|------|

### Table C: Regime-based model performance

| regime | model | avg_MAE | avg_WAPE | avg_MASE | avg_bias | n_products |
|--------|-------|---------|----------|----------|----------|------------|

### Table D: Hybrid strategy summary

| product_id | regime | selected_model | hybrid_MAE | best_single_model | gain_vs_global_baseline |
|------------|--------|----------------|------------|-------------------|-------------------------|

### Recommended final message under the table

A regime-based hybrid strategy should outperform or match the strongest global baseline while remaining more interpretable and better aligned with the statistical structure of each demand pattern.
